In [ ]:
# run this by itself to run r
%load_ext rpy2.ipython

In [52]:
%%R -o test
library(tidyverse)
train <- read_csv('https://raw.githubusercontent.com/blacktreeM/econ/refs/heads/main/train.csv')
test <- read_csv('https://raw.githubusercontent.com/blacktreeM/econ/refs/heads/main/test.csv')

Rows: 10886 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (11): season, holiday, workingday, weather, temp, atemp, humidity, wind...
dttm  (1): datetime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 6493 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (8): season, holiday, workingday, weather, temp, atemp, humidity, winds...
dttm (1): datetime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [ ]:
%%R 
head(train$datetime)

In [ ]:
%%R 
train$year <- substring(train$datetime, 1, 4) # extract year
train$month <- substring(train$datetime, 6, 7) # extract month
train$hour <- substring(train$datetime, 12, 13) # extract hour
train$day <- weekdays(as.Date(train$datetime))

In [ ]:
%%R
test$year <- substring(test$datetime, 1, 4) # extract year
test$month <- substring(test$datetime, 6, 7) # extract month
test$hour <- substring(test$datetime, 12, 13) # extract hour
test$day <- weekdays(as.Date(test$datetime))

In [ ]:
%%R
table(train$year)
table(train$month)
table(train$day)

table(test$year)
table(test$month)
table(test$day)

In [ ]:
%%R
train$day <- weekdays(as.Date(train$datetime))
table(train$day)

In [ ]:
%%R
aggregate(count~day, mean, data=train)

In [ ]:
%%R
train$hour <- substring(train$datetime, 12, 13) # extract hour
table(train$hour)

In [ ]:
%%R
plot(aggregate(count~hour, mean, data=train), type= 'h')

In [ ]:
%%R
# 2/18 assignment
lm(count~hour+day, data=train)

In [ ]:
%%R
# Fit the model
my_model <- lm(count ~ year + month + hour + day + temp + humidity + windspeed +
               factor(season) + factor(weather) + holiday, data = train)
my_model

# Predict, then clip negatives to zero
test$count <- predict(my_model, test)
test$count <- ifelse(test$count < 0, 0, test$count)

summary(test$count)

In [ ]:
%%R
# convert neg numbers to zeros
test$count <- ifelse(test$count<0, 0, test$count)
test$count <- predict(my_model, test)
summary(test$count)

In [ ]:
%%R
submission = subset(test, select = c(datetime, count))
head(submission)

In [50]:
%%R
submission <- subset(test, select = c(datetime, count))

# Strip timezone info by converting to plain string
submission$datetime <- format(submission$datetime, "%Y-%m-%d %H:%M:%S")

write.csv(submission, 'C:/Users/tyler/OneDrive/Documents/GitHub/work_to_show/Classes Spring 2026/Econometrics/parker.csv', row.names = F)

# Verify it looks right
head(submission)

# A tibble: 6 × 2
  datetime             count
  <chr>                <dbl>
1 2011-01-20 00:00:00  -77.1
2 2011-01-20 01:00:00  -79.1
3 2011-01-20 02:00:00  -90.4
4 2011-01-20 03:00:00 -108. 
5 2011-01-20 04:00:00 -109. 
6 2011-01-20 05:00:00 -102. 


In [53]:
print(test.head())

                   datetime  season  holiday  workingday  weather   temp  \
1 2011-01-20 00:00:00+00:00     1.0      0.0         1.0      1.0  10.66   
2 2011-01-20 01:00:00+00:00     1.0      0.0         1.0      1.0  10.66   
3 2011-01-20 02:00:00+00:00     1.0      0.0         1.0      1.0  10.66   
4 2011-01-20 03:00:00+00:00     1.0      0.0         1.0      1.0  10.66   
5 2011-01-20 04:00:00+00:00     1.0      0.0         1.0      1.0  10.66   

    atemp  humidity  windspeed  
1  11.365      56.0    26.0027  
2  13.635      56.0     0.0000  
3  13.635      56.0     0.0000  
4  12.880      56.0    11.0014  
5  12.880      56.0    11.0014  
